In [2]:
import numpy as np
import pandas as pd
import yaml
import requests

In [4]:
member = pd.read_parquet('../data/raw/member_details.parquet')

In [7]:
member.head(3).T

,0,1,2
bioguideId,W000832,B001328,G000607
birthYear,1987,1992,1981
currentMember,True,True,True
directOrderName,Aisha Wahab,Everton Blair Jr.,James Gallagher
district,14.0,13.0,1.0
firstName,Aisha,Everton,James
honorificName,Ms.,Mr.,Mr.
invertedOrderName,"Wahab, Aisha","Blair, Everton","Gallagher, James"
lastName,Wahab,Blair,Gallagher
officialWebsiteUrl,https://wahab.house.gov/,https://blair.house.gov/,https://gallagher.house.gov


In [10]:
voteview_url = 'https://voteview.com/static/data/out/members/HS119_members.csv'
ideology = pd.read_csv(voteview_url)
ideology.head(3).T

,0,1,2
congress,119,119,119
chamber,President,House,House
icpsr,99912,20301,21102
state_icpsr,99,41,41
district_code,0,3,7
state_abbrev,USA,AL,AL
party_code,200,200,100
occupancy,0.0,NaN,NaN
last_means,0.0,NaN,NaN
bioname,"TRUMP, Donald John","ROGERS, Mike Dennis","SEWELL, Terri"


In [18]:
## the not so equivalent - equivalent to a SQL join

## Five kinds of merge/join
## 1. Inner join: drop any row that has no match in the other dataset
## 2. Left join: keeps all rows in the first dataset listed, drops unmatched rows in the second
## 3. Right join: keeps all rows in the second dataset, drops unmatched rows in the first
## 4. Full/outer join: keeps all rows
## 5. Cross join: all pairwise combinations -- explodes your data

## Relationships
## L and R are my two datasets
## Fill in these sentences:
## "One row in L should match with A={one; many} row(s) in R"
## "One row in R should match with B={one; many} row(s) in L"
## This join is a B-to-A

## ex. L = committees, one row per committee
## R = committee membership, one row per member/committee assignment
## One row in committees should match with many rows in assignment
## One row in assignment should match to one row in committees

member_ideology = pd.merge(member, ideology,
                           left_on = 'bioguideId',
                           right_on = 'bioguide_id',
                           how = 'outer', ## the default is 'inner', but we recommend using outer / full join, will show us what matched and what failed to match
                           indicator = 'matched', ## new column to tell you what matched and what didn't
                           validate = 'one_to_many') ## run through the expectation in 'Relationships' above

In [19]:
ideology['bioguide_id'].value_counts()

bioguide_id
K000401    2
R000575    1
S001185    1
P000609    1
M001212    1
          ..
J000312    1
B001230    1
J000293    1
L000571    1
B001261    1
Name: count, Length: 555, dtype: int64

In [20]:
ideology.query("bioguide_id == 'K000401'").T

,60,78
congress,119,119
chamber,House,House
icpsr,22336,92336
state_icpsr,71,71
district_code,3,3
state_abbrev,CA,CA
party_code,200,328
occupancy,NaN,NaN
last_means,NaN,NaN
bioname,"KILEY, Kevin","KILEY, Kevin"


In [21]:
member_ideology['matched'].value_counts()

matched
both          556
right_only      1
left_only       0
Name: count, dtype: int64

In [23]:
member_ideology.query("matched == 'right_only'").T

,556
bioguideId,NaN
birthYear,NaN
currentMember,NaN
directOrderName,NaN
district,NaN
firstName,NaN
honorificName,NaN
invertedOrderName,NaN
lastName,NaN
officialWebsiteUrl,NaN


In [24]:
member_ideology = member_ideology.query("bioname != 'TRUMP, Donald John'")

In [ ]:
member_ideology['matched'].value_counts()

matched
both          556
left_only       0
right_only      0
Name: count, dtype: int64

In [26]:
member_ideology = member_ideology.drop('matched', axis=1)

In [31]:
yaml_url = 'https://github.com/unitedstates/congress-legislators/blob/main/legislators-current.yaml?raw=True'
r = requests.get(yaml_url)
r

<Response [200]>

In [34]:
yaml.safe_load(r.text)

[{'id': {'bioguide': 'C000127',
   'thomas': '00172',
   'lis': 'S275',
   'govtrack': 300018,
   'opensecrets': 'N00007836',
   'votesmart': 27122,
   'fec': ['S8WA00194', 'H2WA01054'],
   'cspan': 26137,
   'wikipedia': 'Maria Cantwell',
   'house_history': 10608,
   'ballotpedia': 'Maria Cantwell',
   'maplight': 544,
   'icpsr': 39310,
   'wikidata': 'Q22250',
   'google_entity_id': 'kg:/m/01x68t',
   'pictorial': 13398},
  'name': {'first': 'Maria',
   'last': 'Cantwell',
   'official_full': 'Maria Cantwell'},
  'bio': {'birthday': '1958-10-13', 'gender': 'F'},
  'terms': [{'type': 'rep',
    'start': '1993-01-05',
    'end': '1995-01-03',
    'state': 'WA',
    'district': 1,
    'party': 'Democrat'},
   {'type': 'sen',
    'start': '2001-01-03',
    'end': '2007-01-03',
    'state': 'WA',
    'class': 1,
    'party': 'Democrat',
    'url': 'http://cantwell.senate.gov'},
   {'type': 'sen',
    'start': '2007-01-04',
    'end': '2013-01-03',
    'state': 'WA',
    'class': 1,
    